# CUDA backend

Requires an NVIDIA GPU and a compatible driver. From `simulation`, run `uv sync --extra notebook --extra cuda`. Restart the kernel and select `simulation/.venv/bin/python`. CPU remains the default backend.

Do not set `NUMBA_ENABLE_CUDASIM` for performance measurements. The simulator is only for testing logic.

In [ ]:
from simulation import create_simulation, run_timed, benchmark

model = create_simulation(
    (100, 100), initial_infected=20, seed=42,
    infection_probability=0.06, recover_probability=0.05,
    backend="cuda",
)

## Compute and record history, then visualize

This variant downloads the full grid after every step. It preserves the existing API, but transfers may dominate execution time. `compute_seconds` includes transfers and synchronization, rather than just kernel execution.

In [ ]:
measured = run_timed(model, steps=100)
print("Computation including transfers:", measured.compute_seconds, "s")
print("History recording:", measured.recording_seconds, "s")

In [ ]:
measured.result.display(interval=100)

## Batch execution without history

`advance` uploads the grid once per batch and downloads only the final result. `benchmark` leaves the original model unchanged; buffer and RNG initialization take place outside the measured samples. Each sample includes one round-trip transfer and synchronization. The first warmup may include compilation.

The same seed provides reproducibility within the CUDA backend, but does not produce identical CPU and GPU trajectories because the generators and random draw order differ.

In [ ]:
batch_model = create_simulation((256, 256), initial_infected=100, seed=42, backend="cuda")
report = benchmark(batch_model, steps=100, repeats=3)
print(report.milliseconds_per_step, "ms/step (batch)")
batch_model.advance(100)
print("Final state:", batch_model.grid.shape)

## Select initial infections interactively

The optional cells below open the control panel. `steps_per_frame` reduces drawing frequency; recording the full history still requires a transfer after every step.

In [ ]:
%matplotlib widget
from simulation import explore
explorer = explore(grid_shape=(40, 40), backend="cuda", seed=42, steps=200, steps_per_frame=5)